# OpenAPI 扩展与客户端生成

学习目标：为小接口写清机器可读的调用约定，并从 OpenAPI 生成 Python 客户端，核对成功和失败响应。

前置知识：HTTP 请求与响应、OpenAPI 基本结构、Pydantic 模型、Python 函数与异常处理、模块导入和上下文管理器。

环境准备：见 [README.md](README.md)。选择课程环境的 Python 3 (ipykernel)，工作目录为 content/Web与应用开发/FastAPI。

本章使用 FastAPI 的 OpenAPI 3.1.0 文档与 openapi-python-client 0.29.1。生成工具和生成代码都使用课程环境，不另建环境或安装生成包。第 6 节起需要在独立终端运行本地服务，端口为 8220。每次从空内核顺序运行；需要重新生成客户端时，先执行篇末清理单元，再重启内核。

配套脚本：位于 scripts/22-openapi-and-client/。

1. [app.py](scripts/22-openapi-and-client/app.py)：把正文逐步定义的最终接口与文档扩展合并，用于真实网络调用。

## 1 从一个有返回模型的接口开始

OpenAPI 用结构化文档描述路径、参数和响应。生成客户端的工具读取这份文档，再写出调用函数与数据类。先让一个接口返回编号和标题，使用 response_model 明确成功响应结构。

这里的 basic_app 只演示最初的成功分支；后面会重新创建 app，加入记录不存在的分支。

In [1]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel


class Record(BaseModel):
    id: int
    title: str


basic_app = FastAPI(title="Study API", version="1.0.0")


@basic_app.get("/records/{record_id}", response_model=Record, operation_id="read_record")
def read_basic_record(record_id: int) -> Record:
    return Record(id=record_id, title="学习 OpenAPI")


with TestClient(basic_app) as client:
    response = client.get("/records/1")
assert response.status_code == 200
print(response.json())  # 预期：{'id': 1, 'title': '学习 OpenAPI'}。

{'id': 1, 'title': '学习 OpenAPI'}


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2 给操作稳定的名称

operationId 是 OpenAPI 中标识一次接口操作的名称，在整份 API 文档中必须唯一，并且区分大小写。FastAPI 装饰器使用 operation_id 参数设置它；本章固定为 read_record，避免函数名或路径调整时无意改变生成客户端的调用入口。

下面只查看该 GET 操作。响应中的 $ref 指向文档内的组件；这里是 components/schemas 下的 Record 模式，描述对象有哪些字段。

In [2]:
basic_schema = basic_app.openapi()
operation = basic_schema["paths"]["/records/{record_id}"]["get"]
record_schema = basic_schema["components"]["schemas"]["Record"]

print("operationId:", operation["operationId"])  # 预期：operationId: read_record。
# 预期：200 模式: {'$ref': '#/components/schemas/Record'}。
print("200 模式:", operation["responses"]["200"]["content"]["application/json"]["schema"])
print("必填字段:", record_schema["required"])  # 预期：必填字段: ['id', 'title']。
assert operation["operationId"] == "read_record"
assert set(record_schema["required"]) == {"id", "title"}

operationId: read_record
200 模式: {'$ref': '#/components/schemas/Record'}
必填字段: ['id', 'title']


## 3 补充应用和分组元数据

元数据说明接口的名称和用途。title 与 version 描述这份 API；version 是应用自己声明的版本，不是 FastAPI 的版本。openapi_tags 为分组补充说明，路由上的 tags 将操作归入相应分组。

现在创建最终的 app，继续复用上面定义的 Record。应用标题固定为 Study API，后面生成的包采用对应的 Python 名称 study_api_client。

In [3]:
app = FastAPI(
    title="Study API",
    version="1.0.0",
    description="查询一条用于学习的记录。",
    openapi_tags=[{"name": "records", "description": "学习记录查询"}],
)

print(app.title, app.version)  # 预期：Study API 1.0.0。
print(app.openapi_tags)  # 分组的 name 将与下一段路由的 tags 对应。

Study API 1.0.0
[{'name': 'records', 'description': '学习记录查询'}]


## 4 声明失败响应与操作扩展

查询不存在的记录时，接口会返回 404。responses 中的 model 是 FastAPI 的配置写法：它把 Pydantic 模型转成 OpenAPI 响应模式。声明 404 不会自动实现失败分支，也不会替你校验异常处理器返回的内容；下面由 HTTPException 产生实际响应。

summary 是操作的简短说明。openapi_extra 会与已有操作文档深度合并；这里添加 x-owner 作为示例维护方标记。OpenAPI 自定义扩展字段以 x- 开头，具体工具可以选择是否处理它，它不会改变接口的运行逻辑。

In [4]:
class Problem(BaseModel):
    detail: str


# operation_id 供客户端生成识别操作，responses 另行描述 404 的结构。
@app.get(
    "/records/{record_id}",
    response_model=Record,
    operation_id="read_record",
    tags=["records"],
    summary="按编号查询记录",
    responses={404: {"model": Problem, "description": "记录不存在"}},
    openapi_extra={"x-owner": "learning-team"},
)
def read_record(record_id: int) -> Record:
    if record_id != 1:
        raise HTTPException(status_code=404, detail="record not found")
    return Record(id=1, title="学习 OpenAPI")


# 对照声明与实际响应，避免只在文档中声明一个并未实现的错误模型。
with TestClient(app) as client:
    found = client.get("/records/1")
    missing = client.get("/records/99")
assert found.status_code == 200 and missing.status_code == 404
assert Problem.model_validate(missing.json()).detail == "record not found"
print(found.status_code, found.json())  # 预期：200 {'id': 1, 'title': '学习 OpenAPI'}。
print(missing.status_code, missing.json())  # 实际 404 与声明的 Problem 结构一致。

200 {'id': 1, 'title': '学习 OpenAPI'}
404 {'detail': 'record not found'}


用同一份文档查看 200、404 和扩展字段。响应码在 OpenAPI JSON 中是字符串键；$ref 最后一个名称能帮助我们辨认各状态对应的模型。

In [5]:
operation = app.openapi()["paths"]["/records/{record_id}"]["get"]
for status in ("200", "404"):
    response_schema = operation["responses"][status]["content"]["application/json"]["schema"]
    # 预期：200 指向 #/components/schemas/Record；404 指向 #/components/schemas/Problem。
    print(status, response_schema["$ref"])
print("分组与维护方:", operation["tags"], operation["x-owner"])  # 预期：['records'] learning-team。
assert operation["responses"]["404"]["description"] == "记录不存在"

200 #/components/schemas/Record
404 #/components/schemas/Problem
分组与维护方: ['records'] learning-team


## 5 定制整份文档并缓存

操作级扩展放在 openapi_extra；需要修改整份文档时，可以用 get_openapi 根据 app.routes 生成文档，再替换 app.openapi。下面向 info 添加 x-course，仍然保留前面的应用元数据和分组。

app.openapi_schema 保存生成结果，避免每次请求都重新生成。前面已经查看过文档，因此替换方法前清空旧缓存。所有路由应先注册完；后续改变影响文档的配置时，也要清空缓存后重新生成。

In [6]:
from fastapi.openapi.utils import get_openapi


def custom_openapi() -> dict:
    if app.openapi_schema is not None:
        return app.openapi_schema
    schema = get_openapi(
        title=app.title,
        version=app.version,
        description=app.description,
        routes=app.routes,
        tags=app.openapi_tags,
    )
    schema["info"]["x-course"] = "FastAPI"
    app.openapi_schema = schema
    return schema


app.openapi_schema = None
app.openapi = custom_openapi
schema = app.openapi()
# 预期：{'title': 'Study API', 'description': '查询一条用于学习的记录。', 'version': '1.0.0', 'x-course': 'FastAPI'}。
print(schema["info"])
print("复用缓存:", schema is app.openapi())  # 预期：复用缓存: True。

{'title': 'Study API', 'description': '查询一条用于学习的记录。', 'version': '1.0.0', 'x-course': 'FastAPI'}
复用缓存: True


清空缓存只影响文档生成，不会删除路由。下面再次生成，比较文档内容与对象身份，并检查所有已声明操作的 operationId 没有重复。methods 列出本次要检查的 HTTP 方法名。

In [7]:
app.openapi_schema = None
regenerated = app.openapi()
assert regenerated == schema and regenerated is not schema

methods = {"get", "put", "post", "delete", "options", "head", "patch", "trace"}
operation_ids = [
    operation["operationId"]
    for path in regenerated["paths"].values()
    for method, operation in path.items()
    if method in methods
]
assert len(operation_ids) == len(set(operation_ids))
print("重建后内容相同:", regenerated == schema)  # 预期：重建后内容相同: True。
print("唯一操作名:", operation_ids)  # 预期：唯一操作名: ['read_record']。

重建后内容相同: True
唯一操作名: ['read_record']


## 6 获取真实服务发布的 OpenAPI

配套 app.py 合并了上面的 Record、Problem、最终 app、查询路由和 custom_openapi。它不包含 basic_app；导入模块不会启动服务器。

Step 1：在已激活课程环境、工作目录为 content/Web与应用开发/FastAPI 的独立终端启动服务。

```bash
python -m uvicorn app:app --app-dir scripts/22-openapi-and-client --host 127.0.0.1 --port 8220
```

Step 2：终端出现 Application startup complete 后，运行下面的 Code 单元，获取真实服务发布的文档。

In [8]:
import httpx


base_url = "http://127.0.0.1:8220"
with httpx.Client(base_url=base_url, timeout=2.0, trust_env=False) as client:
    response = client.get("/openapi.json")
    response.raise_for_status()
    published_schema = response.json()

# 后面将从这份真实服务文档生成客户端；同时检查配套接口与正文一致。
assert published_schema == app.openapi()
print(response.status_code, published_schema["openapi"])  # 预期：200 3.1.0。
print("发布的操作:", published_schema["paths"]["/records/{record_id}"]["get"]["operationId"])  # 预期：发布的操作: read_record。

200 3.1.0
发布的操作: read_record


## 7 从文档生成 Python 包

openapi-python-client 0.29.1 支持 OpenAPI 3.0 和 3.1，但并不支持全部规范特性。这里选用下列生成选项；出现警告也让命令失败，以便先检查未支持的内容。

| 选项 | 中文含义 | 本章用途 |
| --- | --- | --- |
| --path | 本地文档路径 | 读取刚获取并保存的 OpenAPI JSON |
| --meta none | 不生成项目元数据 | 只生成包源码，不创建安装项目 |
| --output-path | 输出位置 | 写入尚不存在的 study_api_client 目录 |
| --fail-on-warning | 警告时返回失败 | 不忽略生成器报告的问题 |

下面先创建本次实验的临时目录，并保存刚获取的文档。TemporaryDirectory 保留到本章清理单元执行；若生成或调用失败，也先执行该清理单元，再重启内核重试。

In [9]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory


temporary = TemporaryDirectory(prefix="fastapi-client-")
root = Path(temporary.name)
schema_path = root / "openapi.json"
schema_path.write_text(json.dumps(published_schema, ensure_ascii=False), encoding="utf-8")
package_path = root / "study_api_client"
print("已保存文档：", schema_path.name)  # 预期：已保存文档： openapi.json。

已保存文档： openapi.json


现在运行生成命令，直接查看其输出。--meta none 使 0.29.1 把包源码直接写到输出目录，因此目录名使用可导入的 study_api_client。sys.executable 选择当前内核的 Python，60 秒为生成命令的等待上限。

生成器默认调用已安装的 Ruff 整理代码。按环境说明从已激活的终端启动 Notebook，使该命令可以被找到。出现错误或警告时，先查看报告并清理本次临时目录，再修改文档重试。

In [10]:
import os
import subprocess
import sys


result = subprocess.run(
    [sys.executable, "-m", "openapi_python_client", "generate",
     "--path", str(schema_path), "--meta", "none",
     "--output-path", str(package_path), "--fail-on-warning"],
    capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONUTF8": "1"}, timeout=60,
)
print(result.stdout, end="")  # 预期：Generating 后跟本次临时目录中的 study_api_client 路径；随机目录名会变化。
print(result.stderr, end="")  # 预期：正常生成时为空；出现诊断时原样显示，由退出码决定是否失败。
result.check_returncode()
assert (package_path / "api/records/read_record.py").is_file()
print("已生成调用模块：study_api_client/api/records/read_record.py")  # 预期：确认 read_record.py 已生成后显示该模块路径。

Generating C:\Users\ZHUANG\AppData\Local\Temp\fastapi-client-4w8g7mb9\study_api_client
已生成调用模块：study_api_client/api/records/read_record.py


## 8 导入并调用生成的客户端

本章的 records 分组和 read_record 操作生成 api/records/read_record.py；响应模式生成 models 下的 Record、Problem 等数据类。先将临时目录加入 Python 的模块搜索路径，再普通导入这些名称。

这里只增加本次生成目录，清理时移除。生成包已经导入后，重新生成文件不会自动替换内核中的模块；修改接口后先清理、重启内核，再从头获取文档和生成代码。

In [11]:
sys.path.insert(0, str(root))

from study_api_client import Client
from study_api_client.api.records import read_record
from study_api_client.models.problem import Problem as ClientProblem
from study_api_client.models.record import Record as ClientRecord

Client 管理 HTTPX 连接；with 退出时关闭连接。sync_detailed 同时保留状态码和解析结果 parsed，sync 则直接返回 parsed。已声明的 404 解析为 Problem；raise_on_unexpected_status=True 针对文档未声明的状态码抛出 UnexpectedStatus。

生成的数据类使用 attrs，并不是服务端的 Pydantic 模型。下面分别核对状态码、模型类型和字段值，并为 HTTPX 设置超时。

In [12]:
with Client(
    base_url=base_url,
    timeout=httpx.Timeout(2.0),
    httpx_args={"trust_env": False},
    raise_on_unexpected_status=True,
) as client:
    found = read_record.sync_detailed(record_id=1, client=client)
    missing = read_record.sync_detailed(record_id=99, client=client)
    assert found.status_code == 200 and isinstance(found.parsed, ClientRecord)
    assert (found.parsed.id, found.parsed.title) == (1, "学习 OpenAPI")
    assert missing.status_code == 404 and isinstance(missing.parsed, ClientProblem)
    assert missing.parsed.detail == "record not found"
    # 预期：200 Record {'id': 1, 'title': '学习 OpenAPI'}。
    print(int(found.status_code), type(found.parsed).__name__, found.parsed.to_dict())
    # 预期：404 Problem {'detail': 'record not found'}。
    print(int(missing.status_code), type(missing.parsed).__name__, missing.parsed.to_dict())
assert client.get_httpx_client().is_closed
print("生成客户端连接已关闭：", client.get_httpx_client().is_closed)  # 预期：生成客户端连接已关闭： True。

200 Record {'id': 1, 'title': '学习 OpenAPI'}
404 Problem {'detail': 'record not found'}
生成客户端连接已关闭： True


## 9 清理本次生成文件

先移除临时搜索路径，再用 cleanup 删除本次文档和生成代码。生成或调用失败后，也可运行此单元；客户端由上一段的 with 关闭。清理文件后，已导入的模块仍属于当前内核，后续重新生成前重启内核即可。

实际项目中应把生成过程纳入构建步骤，在接口变更后重新生成并测试；不要把手改生成文件作为长期维护方式。

In [13]:
if str(root) in sys.path:
    sys.path.remove(str(root))
temporary.cleanup()
assert not root.exists()
assert str(root) not in sys.path
print("临时文档和生成代码已清理，搜索路径已移除")  # 预期：临时文档和生成代码已清理，搜索路径已移除。

临时文档和生成代码已清理，搜索路径已移除


Step 3：调用完成后，在运行 Uvicorn 的终端按 Ctrl+C，确认出现 Application shutdown complete 并返回命令提示符。

## 本章小结

（1）响应模型提供可生成代码的结构；operationId 需要唯一，名称稳定也有助于保持客户端入口稳定。

（2）元数据、附加响应和 x- 扩展描述接口。实际返回内容仍要与声明一致，扩展能否被工具处理取决于工具支持。

（3）custom_openapi 可以修改整份文档；缓存避免重复生成，修改路由或文档配置后需要重新生成。

（4）生成成功只是第一步。本章从真实服务文档生成客户端，并实际检查了 200 的 Record 和 404 的 Problem，以及客户端和临时文件的清理。

## 练习

1. 为 Record 增加必填的 completed: bool，并让编号 1 的响应返回 False。同步修改配套服务并重启，再生成客户端；检查响应模式的 required 包含 completed，生成对象的 completed 为 False。

2. 将 operation_id 改为 get_record，同步正文、服务和调用代码。重新生成，检查新的调用模块为 api/records/get_record.py，200 与 404 调用仍然通过。

3. 将不存在记录的错误改为包含 code: str 和 detail: str，令 code 为 RECORD_NOT_FOUND。同步 Problem、异常返回逻辑和文档声明；生成客户端后检查两个字段都存在且值正确。

4. 在 custom_openapi 中增加 info 下的 x-audience，值为 local-learners。清空缓存后重新读取文档，确认该字段存在，同时原来的 operationId、200 和 404 模式保持一致。

### 提示

- 第 1、2 题改变服务代码后先正常关闭并重新启动服务，获取新文档再生成，避免使用旧进程。
- 第 3 题可以使用 JSONResponse 明确返回自定义错误体；只改 responses 的 model 不会改变 HTTPException 的默认响应结构。
- 第 4 题通过文档字段和原有调用检查扩展；无需假定生成器会将自定义扩展变成客户端属性。

每次修改服务后，先执行本章清理单元并重启内核，再从头运行；这样导入的是新生成的包。

## 参考与引用来源

1. **FastAPI 官方文档**：[响应模型](https://fastapi.tiangolo.com/tutorial/response-model/)、[错误处理](https://fastapi.tiangolo.com/tutorial/handling-errors/)与[测试](https://fastapi.tiangolo.com/tutorial/testing/)支持最小接口和实际响应核对；[元数据](https://fastapi.tiangolo.com/tutorial/metadata/#metadata-for-api)及 Metadata for tags、[操作配置](https://fastapi.tiangolo.com/advanced/path-operation-advanced-configuration/#openapi-operationid)的 operationId、OpenAPI Extra 和 OpenAPI Extensions、[附加响应](https://fastapi.tiangolo.com/advanced/additional-responses/#additional-response-with-model)支持接口声明；[扩展 OpenAPI](https://fastapi.tiangolo.com/how-to/extending-openapi/#the-normal-process)的生成、缓存和替换方法，以及[生成 SDK](https://fastapi.tiangolo.com/advanced/generate-clients/)支持文档到客户端的流程。

2. **OpenAPI Initiative 规范站**：[OpenAPI 3.1.0 §4.8.10 Operation Object](https://spec.openapis.org/oas/v3.1.0.html#operation-object)说明 operationId 的唯一性与大小写要求；[§4.9 Specification Extensions](https://spec.openapis.org/oas/v3.1.0.html#specification-extensions)说明 x- 扩展及工具支持边界。

3. **GitHub / openapi-generators 第一方源码**：[openapi-python-client v0.29.1 README](https://github.com/openapi-generators/openapi-python-client/blob/v0.29.1/README.md)的 Usage、What You Get、generate_all_tags、post_hooks；[生成入口源码](https://github.com/openapi-generators/openapi-python-client/blob/v0.29.1/openapi_python_client/__init__.py)与[CLI 源码](https://github.com/openapi-generators/openapi-python-client/blob/v0.29.1/openapi_python_client/cli.py)对应 --meta none、输出目录与警告处理；[Client 模板](https://github.com/openapi-generators/openapi-python-client/blob/v0.29.1/openapi_python_client/templates/client.py.jinja)、[操作模块模板](https://github.com/openapi-generators/openapi-python-client/blob/v0.29.1/openapi_python_client/templates/endpoint_module.py.jinja)和[模型模板](https://github.com/openapi-generators/openapi-python-client/blob/v0.29.1/openapi_python_client/templates/model.py.jinja)说明连接关闭、状态码解析与 attrs 数据类。本章同时核对已安装 0.29.1 的 CLI 帮助和对应源码。

4. **Python 3.12 官方文档**：[subprocess.run](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)和 CompletedProcess.check_returncode、[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)、[sys.path](https://docs.python.org/3.12/library/sys.html#sys.path)、[模块缓存](https://docs.python.org/3.12/reference/import.html#the-module-cache)支持生成命令、临时目录的 cleanup、搜索路径和重新生成前重启内核的说明。

5. **HTTPX 官方文档**：[Clients](https://www.python-httpx.org/advanced/clients/)的上下文管理器、[Timeouts](https://www.python-httpx.org/advanced/timeouts/)的超时范围与[Environment Variables](https://www.python-httpx.org/environment_variables/)的 trust_env 支持本地请求与显式超时。

6. **Uvicorn 官方文档**：[Settings](https://www.uvicorn.org/settings/#application)的应用入口、app-dir 与 Socket binding 支持配套服务启动；[Server Behavior](https://www.uvicorn.org/server-behavior/#graceful-process-shutdown)说明正常关闭。